### Minicurso Sistemas Multi-agente com LangGraph

## Sistema Multi Agente 

Moacir Antonelli Ponti - 2025

---

In [ ]:
from typing import TypedDict, Optional, Dict, Any
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv

load_dotenv()

Vamos propor um sistema multi-agente que tenha 2 LLMs:
1) É um gerador de questões de múltipla-escolha
2) É um avaliador de questões

In [ ]:
class ExamState(TypedDict):
    # ?

In [ ]:
llm_generator = ChatOpenAI(model="gpt-4o-mini")
llm_reviewer  = ChatOpenAI(model="gpt-4o")

In [ ]:
def generate_question(state: ExamState) -> ExamState:
    refinement_prompt = f"""
    Você um professor capaz de gerar questões de múltipla escolha para exames.
    Gere uma questão original considerando os seguintes parâmetros:
    Formato obrigatório:
    Enunciado
    A) ...
    B) ...
    C) ...
    D) ...
    Gabarito: X
    """
    ###
    return state

In [ ]:
def review_question(state: ExamState) -> ExamState:
    prompt = f"""
    Você um professor extremamente rigoroso na avaliação de questões de múltipla escolha para exames.

    Considere a seguinte questão proposta:
    {state['draft_question']}

    Avalie os seguintes pontos:
    - clareza do enunciado no tópico ({state['topic']})
    - ausência de ambiguidade
    - alternativas plausíveis
    - gabarito coerente
    - nível adequado (deveria ser {state['difficulty']})

    Caso seja fora do tópico, já retorne "request_changes".

    Responda no seguinte formato ReAct:

    Thought: analise profundamente
    Action:
      - "approve" se a questão está perfeita
      - "request_changes" se ela contém erros
    Feedback: explique detalhadamente por quê
    """

    return state


In [ ]:
graph = StateGraph(ExamState)

graph.add_node("generator", generate_question)
graph.add_node("reviewer", review_question)

graph.set_entry_point("generator")

# Fluxo padrão
graph.add_edge("generator", "reviewer")

# Loop ReAct: se reprovado, volta para o gerador
def should_retry(state: ExamState):
    return not state["approved"]

graph.add_conditional_edges(
    "reviewer",
    should_retry,
    {
        True: "generator",  # volta para refazer
        False: END          # termina
    }
)

app = graph.compile()


In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = {
    "topic": "Física / Leis de Newton",
    "difficulty": "médio",
    "draft_question": "",
    "review_feedback": "",
    "approved": False
}

result = app.invoke(initial_state)

In [ ]:

print("Questão final aprovada:\n", result["draft_question"])
print("\nFeedback final do avaliador:\n", result["review_feedback"])